## BUILDING RAG ARCHITECTURES IN 10 JOURS ZERO - TO - MASTERING

#### STEPS OF TECHNIQUES CHUNKING

Ici vous découvrirez les meilleurs techniques pour découper correctement un document(pdf, docx) avant le transmettre à votre LLM. Ce sont les 06 techniques connues qui vous permettront de construire un RAG robustes et autonomes.

### 0- INSTALLATION OF LIBRAIRIES

In [6]:
%uv add langchain langchain_community llama-index
%uv add pypdf2

/home/donerick/Road-to-data-scientist-modern/.venv/bin/python: No module named uv


Note: you may need to restart the kernel to use updated packages.
/home/donerick/Road-to-data-scientist-modern/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


### 1- TECHNIQUES DE FIXE- SIZE CHUNKING

In [7]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import os

/tmp/ipykernel_11973/954994464.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


In [8]:
""" Avec la librairie langchain """

docs = """Machine learning is a subset of artificial intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.
At its core, machine learning involves training algorithms on data to develop models that can perform specific tasks. These models learn relationships within the data, allowing them to generalize and make predictions or decisions on new, unseen data.
There are three primary types of machine learning:
1. Supervised Learning: In supervised learning, the algorithm learns from labeled data, meaning each data point is associated with a correct output or label. The goal is to learn a mapping function that can predict outputs for new inputs.
2. Unsupervised Learning: Unsupervised learning deals with unlabeled data, where the algorithm must discover patterns and structures on its own. Common tasks include clustering, dimensionality reduction, and association rule learning
3. Reinforcement Learning: Reinforcement learning involves an agent learning to make decisions by interacting with an environment. The agent receives rewards or penalties based on its actions, and its goal is to maximize cumulative reward over time.
Machine learning has applications across numerous industries, including healthcare, finance, retail, and manufacturing. From powering recommendation systems and fraud detection to enabling self-driving cars and medical diagnostics, 
machine learning continues to drive innovation and transform how we interact with technology and the world around us.
"""

splitter = CharacterTextSplitter(
    chunk_size=500 ,# taille maximale de tokens , vous pouvez en choisir autre
    chunk_overlap=100 ,# 20% de chevauchement pour éviter de perdre du contexte
    separator="\n", # separateur de chunk 
    length_function=len,
    is_separator_regex=False
)

chunks = splitter.split_text(docs)
# afficher les chuncks

for i, chunk in enumerate(chunks):
    print(f"=====Chunk {i+1}: {chunk}===========")

#### avec un document pdf
"""
Utiliser langchain pour decouper le document pdf en different chuncks
toujour avec la technique de Fixed-Size Chuncking
"""

loader = PyPDFLoader("../data/fairmlbook.pdf")

document = loader.load()
splitter = CharacterTextSplitter(
    chunk_size=500 ,# taille maximale de tokens , vous pouvez en choisir autre
    chunk_overlap=100 ,# 20% de chevauchement pour éviter de perdre du contexte
    )

chunks = splitter.split_documents(document)

print(len(document))
print(len(chunks))
# afficher les chuncks
for i, chunk in enumerate(chunks):
    print(f"=====Chunk {i+1}: {chunk}===========")
    print(chunk.page_content[:500])
    print("\n \n", chunk.metadata)



"""avec la librairie llamaindex"""



=====Chunk 1: Machine learning is a subset of artificial intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.
At its core, machine learning involves training algorithms on data to develop models that can perform specific tasks. These models learn relationships within the data, allowing them to generalize and make predictions or decisions on new, unseen data.
There are three primary types of machine learning:===========
=====Chunk 2: There are three primary types of machine learning:
1. Supervised Learning: In supervised learning, the algorithm learns from labeled data, meaning each data point is associated with a correct output or label. The goal is to learn a mapping function that can predict outputs for new inputs.===========
=====Chunk 3: 2. Unsupervised Learning: Unsupervised learning deals with unlabeled data, where the algorithm must discover patterns and structures on its own. Common tasks include clust

'avec la librairie llamaindex'

### 2- TECHNIQUES DE RECURSIVE CHUNKING

![recursive chunking](/assets/bala-rec-chunking.png)

##### Attention

j'utilise toujours le document pdf charger depuis le debut , donc il faudra faire attentiion à quel splitter vous voulez utiiliser : 

* 1- split_text : pour les chaines de characters 
* 2- split_document: pour les documents avce conservation de leur metadonnées (metadata)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter_recursive = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    is_separator_regex=False
)

docs_chunks = splitter_recursive.split_documents(document)

print(len(document))
print(len(docs_chunks))


for i, chunk in enumerate(docs_chunks):
    print(f"=====Chunk {i+1}: {chunk}===========")
    print(chunk.page_content[:500])
    print("\n \n", chunk.metadata)


294
1089
=====Chunk 1: page_content='FAIRNESS AND MACHINE LEARNING
Limitations and Opportunities
Solon Barocas, Moritz Hardt, Arvind Narayanan
https://fairmlbook.org/ Compiled on Wed Dec 13 14:44:57 CET 2023.' metadata={'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-13T14:45:07+01:00', 'author': 'Solon Barocas, Moritz Hardt, Arvind Narayanan', 'title': 'Fairness and Machine Learning', 'subject': '', 'keywords': '', 'moddate': '2023-12-13T14:45:07+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2021) kpathsea version 6.3.3', 'source': '../data/fairmlbook.pdf', 'total_pages': 294, 'page': 0, 'page_label': 'i'}===========
FAIRNESS AND MACHINE LEARNING
Limitations and Opportunities
Solon Barocas, Moritz Hardt, Arvind Narayanan
https://fairmlbook.org/ Compiled on Wed Dec 13 14:44:57 CET 2023.

 
 {'producer': 'pdfTeX-1.40.22', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-13T14:

### 3  SEMANTIC CHUNKING (SEGMENTATION SEMANTIQUE)

In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding


embed_model = OpenAIEmbedding(
    model_name="BAAI/bge-m3"
)

splitter = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=embed_model,
)

nodes = splitter.get_nodes_from_documents(document)

print(f"Nombre de chunks : {len(nodes)}")

for i, node in enumerate(nodes[:5]):
    print(f"\n===== Chunk {i + 1} =====")
    print(node.get_content()[:500])

### 4- DOCUMENT-BASED CHUNKING

#### A- With FRAMEWORK: LLAMA INDEX
Cette fois-ci j'ai utilisé le framework **llama-index**

visitez la documentation pour en savoir plus :
([llamaindex-documentation](https://www.llamaindex.ai/))

In [ ]:
from llama_index.core.node_parser import MarkdownNodeParser
from pathlib import Path
from llama_index.readers.file import FlatReader


md_docs = FlatReader().load_data(Path("README.md"))
parser = MarkdownNodeParser()

nodes = parser.get_nodes_from_documents(md_docs)
nodes[0].text

#### B- WITH LANGCHAIN FRAMEWORK

In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = """
# Code des douanes

## Dispositions générales

Le présent code définit les règles applicables
aux opérations douanières.

### Article 1

Les marchandises importées sont soumises
aux dispositions du présent code.

### Article 2

Certaines marchandises peuvent bénéficier
d'exonérations sous certaines conditions.

## Droits et taxes

Les marchandises importées peuvent être
soumises à différents droits et taxes.
"""

headers_to_split_on = [
    ("#", "Section"),
    ("##", "Chapter"),
    ("###", "Article"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

chunks = splitter.split_text(markdown_text)

for i, chunk in enumerate(chunks):

    print(f"\n===== Chunk {i + 1} =====")

    print(chunk.page_content)

    print("\nMetadata:")
    print(chunk.metadata)


===== Chunk 1 =====
Le présent code définit les règles applicables
aux opérations douanières.

Metadata:
{'Section': 'Code des douanes', 'Chapter': 'Dispositions générales'}

===== Chunk 2 =====
Les marchandises importées sont soumises
aux dispositions du présent code.

Metadata:
{'Section': 'Code des douanes', 'Chapter': 'Dispositions générales', 'Article': 'Article 1'}

===== Chunk 3 =====
Certaines marchandises peuvent bénéficier
d'exonérations sous certaines conditions.

Metadata:
{'Section': 'Code des douanes', 'Chapter': 'Dispositions générales', 'Article': 'Article 2'}

===== Chunk 4 =====
Les marchandises importées peuvent être
soumises à différents droits et taxes.

Metadata:
{'Section': 'Code des douanes', 'Chapter': 'Droits et taxes'}


### 5- HIERACHICAL CHUNKING

In [ ]:
from langchain_core import document_loaders
from llama_index.core.node_parser import HierarchicalNodeParser

node_parser = HierarchicalNodeParser.from_defaults(
        chunk_sizes=[512, 254, 128]
    )

nodes = node_parser.get_nodes_from_documents(documents)
nodes[0].text

### AGENTIC OU LLM-BASED CHUNKIG